# Comprehensive Recommendation Systems

A complete implementation of various recommendation algorithms including collaborative filtering, content-based filtering, matrix factorization, deep learning approaches, and hybrid systems.

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, lil_matrix
from scipy.spatial.distance import cosine, euclidean
from scipy.stats import pearsonr
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from typing import Dict, List, Tuple, Optional, Union, Any
from dataclasses import dataclass, field
import warnings
import time
from collections import defaultdict
from tqdm import tqdm
import pickle
import json
from pathlib import Path

# Additional libraries for advanced methods
try:
    import implicit
    from implicit.als import AlternatingLeastSquares
    from implicit.bpr import BayesianPersonalizedRanking
    IMPLICIT_AVAILABLE = True
except ImportError:
    print("implicit library not available. Run: pip install implicit")
    IMPLICIT_AVAILABLE = False

try:
    from surprise import Dataset, Reader, SVD, SVDpp, NMF as SurpriseNMF
    from surprise import KNNBasic, KNNWithMeans, KNNWithZScore, KNNBaseline
    from surprise.model_selection import cross_validate
    SURPRISE_AVAILABLE = True
except ImportError:
    print("surprise library not available. Run: pip install surprise")
    SURPRISE_AVAILABLE = False

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Collaborative Filtering

In [ ]:
class CollaborativeFiltering:
    """Collaborative filtering recommendation system."""
    
    def __init__(self, method: str = 'user_based', similarity_metric: str = 'cosine'):
        self.method = method  # 'user_based' or 'item_based'
        self.similarity_metric = similarity_metric
        self.user_item_matrix = None
        self.similarity_matrix = None
        self.user_means = None
        self.item_means = None
        
    def fit(self, interactions: pd.DataFrame):
        """Fit collaborative filtering model."""
        
        # Create user-item matrix
        self.user_item_matrix = interactions.pivot_table(
            index='user_id',
            columns='item_id',
            values='rating',
            fill_value=0
        )
        
        # Calculate means for mean-centered approaches
        self.user_means = self.user_item_matrix.mean(axis=1)
        self.item_means = self.user_item_matrix.mean(axis=0)
        
        # Calculate similarity matrix
        if self.method == 'user_based':
            self.similarity_matrix = self._calculate_similarity(
                self.user_item_matrix.values
            )
        else:  # item_based
            self.similarity_matrix = self._calculate_similarity(
                self.user_item_matrix.values.T
            )
        
        return self
    
    def _calculate_similarity(self, matrix: np.ndarray) -> np.ndarray:
        """Calculate similarity matrix."""
        
        if self.similarity_metric == 'cosine':
            # Add small value to avoid division by zero
            return cosine_similarity(matrix + 1e-9)
        elif self.similarity_metric == 'pearson':
            # Calculate Pearson correlation
            n = matrix.shape[0]
            similarity = np.zeros((n, n))
            
            for i in range(n):
                for j in range(n):
                    if i != j:
                        # Get common items
                        mask = (matrix[i] != 0) & (matrix[j] != 0)
                        if mask.sum() > 0:
                            corr, _ = pearsonr(matrix[i][mask], matrix[j][mask])
                            similarity[i, j] = corr if not np.isnan(corr) else 0
            
            return similarity
        elif self.similarity_metric == 'jaccard':
            # Binary Jaccard similarity
            binary_matrix = (matrix > 0).astype(float)
            intersection = binary_matrix.dot(binary_matrix.T)
            row_sums = binary_matrix.sum(axis=1)
            union = row_sums[:, np.newaxis] + row_sums - intersection
            union[union == 0] = 1  # Avoid division by zero
            return intersection / union
        else:
            raise ValueError(f"Unknown similarity metric: {self.similarity_metric}")
    
    def predict(self, user_id: int, item_id: int) -> float:
        """Predict rating for a user-item pair."""
        
        if self.user_item_matrix is None:
            raise ValueError("Model not fitted yet")
        
        if user_id not in self.user_item_matrix.index:
            return self.item_means[item_id] if item_id in self.item_means else 0
        
        if item_id not in self.user_item_matrix.columns:
            return self.user_means[user_id] if user_id in self.user_means else 0
        
        if self.method == 'user_based':
            return self._predict_user_based(user_id, item_id)
        else:
            return self._predict_item_based(user_id, item_id)
    
    def _predict_user_based(self, user_id: int, item_id: int) -> float:
        """Predict using user-based collaborative filtering."""
        
        user_idx = self.user_item_matrix.index.get_loc(user_id)
        
        # Get similar users who rated this item
        item_ratings = self.user_item_matrix[item_id]
        rated_users = item_ratings[item_ratings != 0].index
        
        if len(rated_users) == 0:
            return self.user_means[user_id]
        
        # Calculate weighted average
        numerator = 0
        denominator = 0
        
        for other_user in rated_users:
            if other_user != user_id:
                other_idx = self.user_item_matrix.index.get_loc(other_user)
                similarity = self.similarity_matrix[user_idx, other_idx]
                
                if similarity > 0:
                    rating = self.user_item_matrix.loc[other_user, item_id]
                    numerator += similarity * rating
                    denominator += abs(similarity)
        
        if denominator == 0:
            return self.user_means[user_id]
        
        return numerator / denominator
    
    def _predict_item_based(self, user_id: int, item_id: int) -> float:
        """Predict using item-based collaborative filtering."""
        
        item_idx = self.user_item_matrix.columns.get_loc(item_id)
        
        # Get items rated by this user
        user_ratings = self.user_item_matrix.loc[user_id]
        rated_items = user_ratings[user_ratings != 0].index
        
        if len(rated_items) == 0:
            return self.item_means[item_id]
        
        # Calculate weighted average
        numerator = 0
        denominator = 0
        
        for other_item in rated_items:
            if other_item != item_id:
                other_idx = self.user_item_matrix.columns.get_loc(other_item)
                similarity = self.similarity_matrix[item_idx, other_idx]
                
                if similarity > 0:
                    rating = self.user_item_matrix.loc[user_id, other_item]
                    numerator += similarity * rating
                    denominator += abs(similarity)
        
        if denominator == 0:
            return self.item_means[item_id]
        
        return numerator / denominator
    
    def recommend(self, user_id: int, n_recommendations: int = 10,
                 exclude_seen: bool = True) -> List[Tuple[int, float]]:
        """Generate recommendations for a user."""
        
        if user_id not in self.user_item_matrix.index:
            # Cold start - return popular items
            popular_items = self.item_means.nlargest(n_recommendations)
            return [(item, score) for item, score in popular_items.items()]
        
        # Get user's rated items
        user_ratings = self.user_item_matrix.loc[user_id]
        
        # Predict ratings for all items
        predictions = {}
        for item_id in self.user_item_matrix.columns:
            if exclude_seen and user_ratings[item_id] != 0:
                continue
            predictions[item_id] = self.predict(user_id, item_id)
        
        # Sort by predicted rating
        sorted_predictions = sorted(predictions.items(), 
                                  key=lambda x: x[1], reverse=True)
        
        return sorted_predictions[:n_recommendations]

## 2. Content-Based Filtering

In [ ]:
class ContentBasedFiltering:
    """Content-based recommendation system."""
    
    def __init__(self, feature_type: str = 'tfidf'):
        self.feature_type = feature_type
        self.item_features = None
        self.user_profiles = None
        self.vectorizer = None
        self.item_similarity = None
        
    def fit(self, item_features: pd.DataFrame, 
           interactions: Optional[pd.DataFrame] = None):
        """Fit content-based model."""
        
        # Process item features
        if self.feature_type == 'tfidf':
            # Assume item_features has a 'description' column
            self.vectorizer = TfidfVectorizer(
                max_features=1000,
                ngram_range=(1, 2),
                stop_words='english'
            )
            self.item_features = self.vectorizer.fit_transform(
                item_features['description'].fillna('')
            )
        else:
            # Assume numerical features
            feature_cols = [col for col in item_features.columns 
                          if col not in ['item_id', 'description']]
            self.item_features = item_features[feature_cols].values
        
        # Calculate item similarity matrix
        self.item_similarity = cosine_similarity(self.item_features)
        
        # Build user profiles if interactions provided
        if interactions is not None:
            self._build_user_profiles(interactions, item_features)
        
        # Store item IDs for reference
        self.item_ids = item_features['item_id'].values
        self.item_id_to_idx = {item_id: idx for idx, item_id in enumerate(self.item_ids)}
        
        return self
    
    def _build_user_profiles(self, interactions: pd.DataFrame, 
                           item_features: pd.DataFrame):
        """Build user profiles based on interaction history."""
        
        self.user_profiles = {}
        
        for user_id in interactions['user_id'].unique():
            user_interactions = interactions[interactions['user_id'] == user_id]
            
            # Get items interacted with and their ratings
            item_ids = user_interactions['item_id'].values
            ratings = user_interactions['rating'].values
            
            # Weight item features by ratings
            weighted_features = np.zeros(self.item_features.shape[1])
            
            for item_id, rating in zip(item_ids, ratings):
                if item_id in self.item_id_to_idx:
                    idx = self.item_id_to_idx[item_id]
                    if hasattr(self.item_features, 'toarray'):
                        weighted_features += rating * self.item_features[idx].toarray().flatten()
                    else:
                        weighted_features += rating * self.item_features[idx]
            
            # Normalize
            if ratings.sum() > 0:
                weighted_features /= ratings.sum()
            
            self.user_profiles[user_id] = weighted_features
    
    def recommend_similar_items(self, item_id: int, 
                               n_recommendations: int = 10) -> List[Tuple[int, float]]:
        """Recommend items similar to a given item."""
        
        if item_id not in self.item_id_to_idx:
            return []
        
        item_idx = self.item_id_to_idx[item_id]
        similarities = self.item_similarity[item_idx]
        
        # Get top similar items (excluding the item itself)
        similar_indices = similarities.argsort()[::-1][1:n_recommendations+1]
        
        recommendations = []
        for idx in similar_indices:
            similar_item_id = self.item_ids[idx]
            similarity_score = similarities[idx]
            recommendations.append((similar_item_id, similarity_score))
        
        return recommendations
    
    def recommend_for_user(self, user_id: int, 
                          n_recommendations: int = 10,
                          exclude_seen: bool = True,
                          seen_items: Optional[List[int]] = None) -> List[Tuple[int, float]]:
        """Recommend items for a user based on their profile."""
        
        if user_id not in self.user_profiles:
            # Cold start - return random popular items
            random_items = np.random.choice(self.item_ids, 
                                          size=min(n_recommendations, len(self.item_ids)),
                                          replace=False)
            return [(item_id, 0.5) for item_id in random_items]
        
        user_profile = self.user_profiles[user_id]
        
        # Calculate similarity between user profile and all items
        if hasattr(self.item_features, 'toarray'):
            item_features_array = self.item_features.toarray()
        else:
            item_features_array = self.item_features
        
        similarities = cosine_similarity([user_profile], item_features_array)[0]
        
        # Sort items by similarity
        item_scores = list(zip(self.item_ids, similarities))
        item_scores.sort(key=lambda x: x[1], reverse=True)
        
        # Filter out seen items if requested
        if exclude_seen and seen_items is not None:
            item_scores = [(item_id, score) for item_id, score in item_scores 
                         if item_id not in seen_items]
        
        return item_scores[:n_recommendations]

## 3. Matrix Factorization

In [ ]:
class MatrixFactorization:
    """Matrix factorization recommendation methods."""
    
    def __init__(self, method: str = 'svd', n_factors: int = 50):
        self.method = method
        self.n_factors = n_factors
        self.model = None
        self.user_factors = None
        self.item_factors = None
        self.user_item_matrix = None
        self.user_id_to_idx = None
        self.item_id_to_idx = None
        self.idx_to_user_id = None
        self.idx_to_item_id = None
        
    def fit(self, interactions: pd.DataFrame):
        """Fit matrix factorization model."""
        
        # Create user-item matrix
        self.user_item_matrix = interactions.pivot_table(
            index='user_id',
            columns='item_id',
            values='rating',
            fill_value=0
        )
        
        # Create mappings
        self.user_id_to_idx = {uid: idx for idx, uid in enumerate(self.user_item_matrix.index)}
        self.item_id_to_idx = {iid: idx for idx, iid in enumerate(self.user_item_matrix.columns)}
        self.idx_to_user_id = {idx: uid for uid, idx in self.user_id_to_idx.items()}
        self.idx_to_item_id = {idx: iid for iid, idx in self.item_id_to_idx.items()}
        
        # Convert to sparse matrix
        sparse_matrix = csr_matrix(self.user_item_matrix.values)
        
        if self.method == 'svd':
            self._fit_svd(sparse_matrix)
        elif self.method == 'nmf':
            self._fit_nmf(sparse_matrix)
        elif self.method == 'als' and IMPLICIT_AVAILABLE:
            self._fit_als(sparse_matrix)
        elif self.method == 'bpr' and IMPLICIT_AVAILABLE:
            self._fit_bpr(sparse_matrix)
        elif self.method == 'surprise_svd' and SURPRISE_AVAILABLE:
            self._fit_surprise_svd(interactions)
        else:
            raise ValueError(f"Method {self.method} not available")
        
        return self
    
    def _fit_svd(self, sparse_matrix):
        """Fit SVD using sklearn."""
        
        svd = TruncatedSVD(n_components=self.n_factors, random_state=42)
        self.user_factors = svd.fit_transform(sparse_matrix)
        self.item_factors = svd.components_.T
        self.model = svd
    
    def _fit_nmf(self, sparse_matrix):
        """Fit NMF using sklearn."""
        
        nmf = NMF(n_components=self.n_factors, init='random', random_state=42)
        self.user_factors = nmf.fit_transform(sparse_matrix)
        self.item_factors = nmf.components_.T
        self.model = nmf
    
    def _fit_als(self, sparse_matrix):
        """Fit ALS using implicit library."""
        
        model = AlternatingLeastSquares(
            factors=self.n_factors,
            regularization=0.01,
            iterations=50,
            random_state=42
        )
        
        # implicit expects item-user matrix
        model.fit(sparse_matrix.T)
        
        self.model = model
        self.user_factors = model.user_factors
        self.item_factors = model.item_factors
    
    def _fit_bpr(self, sparse_matrix):
        """Fit BPR using implicit library."""
        
        model = BayesianPersonalizedRanking(
            factors=self.n_factors,
            learning_rate=0.01,
            regularization=0.01,
            iterations=100,
            random_state=42
        )
        
        # implicit expects item-user matrix
        model.fit(sparse_matrix.T)
        
        self.model = model
        self.user_factors = model.user_factors
        self.item_factors = model.item_factors
    
    def _fit_surprise_svd(self, interactions):
        """Fit SVD using surprise library."""
        
        # Prepare data for surprise
        reader = Reader(rating_scale=(interactions['rating'].min(), 
                                     interactions['rating'].max()))
        data = Dataset.load_from_df(
            interactions[['user_id', 'item_id', 'rating']], 
            reader
        )
        
        # Train SVD
        trainset = data.build_full_trainset()
        model = SVD(n_factors=self.n_factors, random_state=42)
        model.fit(trainset)
        
        self.model = model
        # Note: Surprise doesn't expose factors directly
    
    def predict(self, user_id: int, item_id: int) -> float:
        """Predict rating for user-item pair."""
        
        if self.method == 'surprise_svd' and SURPRISE_AVAILABLE:
            prediction = self.model.predict(user_id, item_id)
            return prediction.est
        
        if user_id not in self.user_id_to_idx:
            return 0
        if item_id not in self.item_id_to_idx:
            return 0
        
        user_idx = self.user_id_to_idx[user_id]
        item_idx = self.item_id_to_idx[item_id]
        
        if self.method in ['als', 'bpr'] and IMPLICIT_AVAILABLE:
            return self.user_factors[user_idx].dot(self.item_factors[item_idx].T)
        else:
            return self.user_factors[user_idx].dot(self.item_factors[item_idx])
    
    def recommend(self, user_id: int, n_recommendations: int = 10,
                 exclude_seen: bool = True) -> List[Tuple[int, float]]:
        """Generate recommendations for a user."""
        
        if self.method == 'surprise_svd' and SURPRISE_AVAILABLE:
            # Predict for all items
            predictions = []
            for item_id in self.user_item_matrix.columns:
                if exclude_seen and self.user_item_matrix.loc[user_id, item_id] != 0:
                    continue
                pred = self.model.predict(user_id, item_id)
                predictions.append((item_id, pred.est))
            
            predictions.sort(key=lambda x: x[1], reverse=True)
            return predictions[:n_recommendations]
        
        if user_id not in self.user_id_to_idx:
            # Cold start
            return []
        
        user_idx = self.user_id_to_idx[user_id]
        
        if self.method in ['als', 'bpr'] and IMPLICIT_AVAILABLE:
            # Use implicit's recommend method
            user_items = self.user_item_matrix.iloc[user_idx].values
            recommendations = self.model.recommend(
                user_idx,
                csr_matrix(user_items),
                N=n_recommendations,
                filter_already_liked_items=exclude_seen
            )
            return [(self.idx_to_item_id[idx], score) 
                   for idx, score in recommendations]
        else:
            # Calculate scores for all items
            user_vector = self.user_factors[user_idx]
            scores = user_vector.dot(self.item_factors.T)
            
            # Get item IDs and scores
            item_scores = [(self.idx_to_item_id[idx], scores[idx]) 
                         for idx in range(len(scores))]
            
            # Filter seen items if requested
            if exclude_seen:
                seen_items = self.user_item_matrix.iloc[user_idx]
                seen_items = seen_items[seen_items != 0].index
                item_scores = [(item_id, score) for item_id, score in item_scores
                             if item_id not in seen_items]
            
            # Sort and return top N
            item_scores.sort(key=lambda x: x[1], reverse=True)
            return item_scores[:n_recommendations]

## 4. Deep Learning Recommenders

In [ ]:
class NeuralCollaborativeFiltering(nn.Module):
    """Neural Collaborative Filtering model."""
    
    def __init__(self, n_users: int, n_items: int, 
                embedding_dim: int = 50, hidden_dims: List[int] = [64, 32, 16]):
        super().__init__()
        
        # Embeddings
        self.user_embedding_mlp = nn.Embedding(n_users, embedding_dim)
        self.item_embedding_mlp = nn.Embedding(n_items, embedding_dim)
        self.user_embedding_gmf = nn.Embedding(n_users, embedding_dim)
        self.item_embedding_gmf = nn.Embedding(n_items, embedding_dim)
        
        # MLP layers
        mlp_layers = []
        input_dim = embedding_dim * 2
        for hidden_dim in hidden_dims:
            mlp_layers.append(nn.Linear(input_dim, hidden_dim))
            mlp_layers.append(nn.ReLU())
            mlp_layers.append(nn.Dropout(0.2))
            input_dim = hidden_dim
        self.mlp = nn.Sequential(*mlp_layers)
        
        # Final prediction layer
        self.prediction = nn.Linear(hidden_dims[-1] + embedding_dim, 1)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize model weights."""
        
        nn.init.normal_(self.user_embedding_mlp.weight, std=0.01)
        nn.init.normal_(self.item_embedding_mlp.weight, std=0.01)
        nn.init.normal_(self.user_embedding_gmf.weight, std=0.01)
        nn.init.normal_(self.item_embedding_gmf.weight, std=0.01)
        
        for module in self.mlp:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
        
        nn.init.xavier_uniform_(self.prediction.weight)
    
    def forward(self, user_ids: torch.Tensor, item_ids: torch.Tensor) -> torch.Tensor:
        """Forward pass."""
        
        # GMF part
        user_embed_gmf = self.user_embedding_gmf(user_ids)
        item_embed_gmf = self.item_embedding_gmf(item_ids)
        gmf_output = user_embed_gmf * item_embed_gmf
        
        # MLP part
        user_embed_mlp = self.user_embedding_mlp(user_ids)
        item_embed_mlp = self.item_embedding_mlp(item_ids)
        mlp_input = torch.cat([user_embed_mlp, item_embed_mlp], dim=-1)
        mlp_output = self.mlp(mlp_input)
        
        # Concatenate and predict
        concat = torch.cat([gmf_output, mlp_output], dim=-1)
        prediction = self.prediction(concat)
        
        return prediction.squeeze()


class DeepRecommenderDataset(Dataset):
    """Dataset for deep learning recommenders."""
    
    def __init__(self, interactions: pd.DataFrame, 
                user_id_to_idx: Dict, item_id_to_idx: Dict):
        self.interactions = interactions
        self.user_id_to_idx = user_id_to_idx
        self.item_id_to_idx = item_id_to_idx
    
    def __len__(self):
        return len(self.interactions)
    
    def __getitem__(self, idx):
        row = self.interactions.iloc[idx]
        user_idx = self.user_id_to_idx[row['user_id']]
        item_idx = self.item_id_to_idx[row['item_id']]
        rating = row['rating']
        
        return {
            'user_id': torch.tensor(user_idx, dtype=torch.long),
            'item_id': torch.tensor(item_idx, dtype=torch.long),
            'rating': torch.tensor(rating, dtype=torch.float)
        }


class DeepLearningRecommender:
    """Deep learning based recommender system."""
    
    def __init__(self, model_type: str = 'ncf', embedding_dim: int = 50):
        self.model_type = model_type
        self.embedding_dim = embedding_dim
        self.model = None
        self.user_id_to_idx = None
        self.item_id_to_idx = None
        self.idx_to_user_id = None
        self.idx_to_item_id = None
        
    def fit(self, interactions: pd.DataFrame, 
           n_epochs: int = 10, batch_size: int = 256,
           learning_rate: float = 0.001):
        """Train deep learning recommender."""
        
        # Create mappings
        unique_users = interactions['user_id'].unique()
        unique_items = interactions['item_id'].unique()
        
        self.user_id_to_idx = {uid: idx for idx, uid in enumerate(unique_users)}
        self.item_id_to_idx = {iid: idx for idx, iid in enumerate(unique_items)}
        self.idx_to_user_id = {idx: uid for uid, idx in self.user_id_to_idx.items()}
        self.idx_to_item_id = {idx: iid for iid, idx in self.item_id_to_idx.items()}
        
        n_users = len(unique_users)
        n_items = len(unique_items)
        
        # Create model
        if self.model_type == 'ncf':
            self.model = NeuralCollaborativeFiltering(
                n_users, n_items, self.embedding_dim
            ).to(device)
        else:
            raise ValueError(f"Unknown model type: {self.model_type}")
        
        # Prepare data
        dataset = DeepRecommenderDataset(
            interactions, self.user_id_to_idx, self.item_id_to_idx
        )
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        
        # Setup training
        optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)
        criterion = nn.MSELoss()
        
        # Training loop
        self.model.train()
        for epoch in range(n_epochs):
            total_loss = 0
            for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}/{n_epochs}"):
                user_ids = batch['user_id'].to(device)
                item_ids = batch['item_id'].to(device)
                ratings = batch['rating'].to(device)
                
                optimizer.zero_grad()
                predictions = self.model(user_ids, item_ids)
                loss = criterion(predictions, ratings)
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            avg_loss = total_loss / len(dataloader)
            print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}")
        
        return self
    
    def predict(self, user_id: int, item_id: int) -> float:
        """Predict rating for user-item pair."""
        
        if user_id not in self.user_id_to_idx:
            return 0
        if item_id not in self.item_id_to_idx:
            return 0
        
        user_idx = self.user_id_to_idx[user_id]
        item_idx = self.item_id_to_idx[item_id]
        
        self.model.eval()
        with torch.no_grad():
            user_tensor = torch.tensor([user_idx], dtype=torch.long).to(device)
            item_tensor = torch.tensor([item_idx], dtype=torch.long).to(device)
            prediction = self.model(user_tensor, item_tensor)
        
        return prediction.item()
    
    def recommend(self, user_id: int, n_recommendations: int = 10,
                 exclude_seen: bool = True,
                 seen_items: Optional[List[int]] = None) -> List[Tuple[int, float]]:
        """Generate recommendations for a user."""
        
        if user_id not in self.user_id_to_idx:
            return []
        
        user_idx = self.user_id_to_idx[user_id]
        
        # Predict for all items
        self.model.eval()
        predictions = []
        
        with torch.no_grad():
            user_tensor = torch.tensor([user_idx] * len(self.item_id_to_idx), 
                                      dtype=torch.long).to(device)
            item_tensor = torch.tensor(list(range(len(self.item_id_to_idx))),
                                      dtype=torch.long).to(device)
            
            batch_size = 1000
            for i in range(0, len(item_tensor), batch_size):
                batch_users = user_tensor[i:i+batch_size]
                batch_items = item_tensor[i:i+batch_size]
                batch_predictions = self.model(batch_users, batch_items)
                predictions.extend(batch_predictions.cpu().numpy())
        
        # Create item-score pairs
        item_scores = [(self.idx_to_item_id[idx], score) 
                      for idx, score in enumerate(predictions)]
        
        # Filter seen items
        if exclude_seen and seen_items:
            item_scores = [(item_id, score) for item_id, score in item_scores
                         if item_id not in seen_items]
        
        # Sort and return top N
        item_scores.sort(key=lambda x: x[1], reverse=True)
        return item_scores[:n_recommendations]

## 5. Hybrid Recommender System

In [ ]:
class HybridRecommenderSystem:
    """Hybrid recommendation system combining multiple approaches."""
    
    def __init__(self):
        self.models = {}
        self.weights = {
            'collaborative': 0.3,
            'content': 0.2,
            'matrix_factorization': 0.3,
            'deep': 0.2
        }
        self.interactions = None
        self.item_features = None
        
    def fit(self, 
           interactions: pd.DataFrame,
           item_features: Optional[pd.DataFrame] = None,
           user_features: Optional[pd.DataFrame] = None,
           fit_collaborative: bool = True,
           fit_content: bool = True,
           fit_matrix_factorization: bool = True,
           fit_deep: bool = True):
        """Fit all recommendation models."""
        
        self.interactions = interactions
        self.item_features = item_features
        
        print("Training hybrid recommender system...")
        
        # Collaborative Filtering
        if fit_collaborative:
            print("Training collaborative filtering...")
            self.models['collaborative'] = CollaborativeFiltering(
                method='item_based',
                similarity_metric='cosine'
            )
            self.models['collaborative'].fit(interactions)
        
        # Content-Based
        if fit_content and item_features is not None:
            print("Training content-based filtering...")
            self.models['content'] = ContentBasedFiltering()
            self.models['content'].fit(item_features, interactions)
        
        # Matrix Factorization
        if fit_matrix_factorization:
            print("Training matrix factorization...")
            self.models['matrix_factorization'] = MatrixFactorization(
                method='svd',
                n_factors=50
            )
            self.models['matrix_factorization'].fit(interactions)
        
        # Deep Learning
        if fit_deep and len(interactions) > 1000:  # Only if enough data
            print("Training deep learning model...")
            self.models['deep'] = DeepLearningRecommender(
                model_type='ncf',
                embedding_dim=50
            )
            self.models['deep'].fit(interactions, n_epochs=5)
        
        print("Hybrid recommender system training complete!")
        return self
    
    def recommend(self, user_id: int, n_recommendations: int = 10,
                 strategy: str = 'weighted', exclude_seen: bool = True) -> List[Tuple[int, float]]:
        """Generate recommendations for a user."""
        
        # Get user's seen items
        seen_items = set()
        if exclude_seen and self.interactions is not None:
            user_interactions = self.interactions[self.interactions['user_id'] == user_id]
            seen_items = set(user_interactions['item_id'].values)
        
        if strategy == 'weighted':
            return self._weighted_hybrid(user_id, n_recommendations, seen_items)
        elif strategy == 'switching':
            return self._switching_hybrid(user_id, n_recommendations, seen_items)
        elif strategy == 'mixed':
            return self._mixed_hybrid(user_id, n_recommendations, seen_items)
        else:
            raise ValueError(f"Unknown strategy: {strategy}")
    
    def _weighted_hybrid(self, user_id: int, n_recommendations: int,
                        seen_items: set) -> List[Tuple[int, float]]:
        """Weighted hybrid recommendation."""
        
        all_recommendations = {}
        
        # Get recommendations from each model
        for model_name, model in self.models.items():
            try:
                if model_name == 'content':
                    recs = model.recommend_for_user(
                        user_id, n_recommendations * 2, 
                        exclude_seen=True, seen_items=list(seen_items)
                    )
                elif model_name == 'deep':
                    recs = model.recommend(
                        user_id, n_recommendations * 2,
                        exclude_seen=True, seen_items=list(seen_items)
                    )
                else:
                    recs = model.recommend(
                        user_id, n_recommendations * 2,
                        exclude_seen=True
                    )
                
                # Add to recommendations with weight
                weight = self.weights.get(model_name, 0.25)
                for item_id, score in recs:
                    if item_id not in seen_items:
                        if item_id not in all_recommendations:
                            all_recommendations[item_id] = 0
                        all_recommendations[item_id] += weight * score
            except:
                # Model failed, continue with others
                pass
        
        # Sort by weighted score
        sorted_recs = sorted(all_recommendations.items(), 
                           key=lambda x: x[1], reverse=True)
        
        return sorted_recs[:n_recommendations]
    
    def _switching_hybrid(self, user_id: int, n_recommendations: int,
                         seen_items: set) -> List[Tuple[int, float]]:
        """Switching hybrid - choose best model based on context."""
        
        # Determine which model to use based on user profile
        user_interactions = self.interactions[self.interactions['user_id'] == user_id]
        n_interactions = len(user_interactions)
        
        if n_interactions < 5:
            # Cold start - use content-based or popularity
            if 'content' in self.models:
                return self.models['content'].recommend_for_user(
                    user_id, n_recommendations,
                    exclude_seen=True, seen_items=list(seen_items)
                )
        elif n_interactions < 20:
            # Few interactions - use collaborative filtering
            if 'collaborative' in self.models:
                return self.models['collaborative'].recommend(
                    user_id, n_recommendations, exclude_seen=True
                )
        else:
            # Many interactions - use matrix factorization or deep learning
            if 'deep' in self.models:
                return self.models['deep'].recommend(
                    user_id, n_recommendations,
                    exclude_seen=True, seen_items=list(seen_items)
                )
            elif 'matrix_factorization' in self.models:
                return self.models['matrix_factorization'].recommend(
                    user_id, n_recommendations, exclude_seen=True
                )
        
        # Fallback to weighted hybrid
        return self._weighted_hybrid(user_id, n_recommendations, seen_items)
    
    def _mixed_hybrid(self, user_id: int, n_recommendations: int,
                     seen_items: set) -> List[Tuple[int, float]]:
        """Mixed hybrid - combine recommendations from different models."""
        
        recommendations = []
        items_per_model = n_recommendations // len(self.models)
        
        for model_name, model in self.models.items():
            try:
                if model_name == 'content':
                    recs = model.recommend_for_user(
                        user_id, items_per_model,
                        exclude_seen=True, seen_items=list(seen_items)
                    )
                elif model_name == 'deep':
                    recs = model.recommend(
                        user_id, items_per_model,
                        exclude_seen=True, seen_items=list(seen_items)
                    )
                else:
                    recs = model.recommend(
                        user_id, items_per_model,
                        exclude_seen=True
                    )
                
                for item_id, score in recs:
                    if item_id not in seen_items:
                        recommendations.append((item_id, score))
                        seen_items.add(item_id)
            except:
                pass
        
        return recommendations[:n_recommendations]
    
    def explain_recommendation(self, user_id: int, item_id: int) -> Dict:
        """Explain why an item was recommended."""
        
        explanations = {}
        
        # Get predictions from each model
        for model_name, model in self.models.items():
            try:
                if hasattr(model, 'predict'):
                    score = model.predict(user_id, item_id)
                    explanations[model_name] = {
                        'score': score,
                        'weight': self.weights.get(model_name, 0.25)
                    }
            except:
                pass
        
        # Add similar items explanation for content-based
        if 'content' in self.models and self.item_features is not None:
            similar_items = self.models['content'].recommend_similar_items(item_id, 5)
            explanations['similar_items'] = similar_items
        
        # Add user's similar users for collaborative
        if 'collaborative' in self.models:
            user_idx = self.models['collaborative'].user_item_matrix.index.get_loc(user_id)
            similarities = self.models['collaborative'].similarity_matrix[user_idx]
            top_similar_users = similarities.argsort()[-6:-1][::-1]
            explanations['similar_users'] = top_similar_users.tolist()
        
        return explanations

## 6. Evaluation Metrics

In [ ]:
class RecommenderEvaluator:
    """Evaluation metrics for recommender systems."""
    
    @staticmethod
    def train_test_split_temporal(interactions: pd.DataFrame, 
                                 test_size: float = 0.2) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Split data based on timestamp."""
        
        if 'timestamp' in interactions.columns:
            interactions = interactions.sort_values('timestamp')
        
        split_idx = int(len(interactions) * (1 - test_size))
        train = interactions.iloc[:split_idx]
        test = interactions.iloc[split_idx:]
        
        return train, test
    
    @staticmethod
    def calculate_rmse(predictions: List[Tuple[int, int, float]], 
                      actuals: pd.DataFrame) -> float:
        """Calculate Root Mean Square Error."""
        
        squared_errors = []
        for user_id, item_id, predicted in predictions:
            actual = actuals[(actuals['user_id'] == user_id) & 
                           (actuals['item_id'] == item_id)]['rating'].values
            if len(actual) > 0:
                squared_errors.append((predicted - actual[0]) ** 2)
        
        if squared_errors:
            return np.sqrt(np.mean(squared_errors))
        return float('inf')
    
    @staticmethod
    def calculate_mae(predictions: List[Tuple[int, int, float]], 
                     actuals: pd.DataFrame) -> float:
        """Calculate Mean Absolute Error."""
        
        absolute_errors = []
        for user_id, item_id, predicted in predictions:
            actual = actuals[(actuals['user_id'] == user_id) & 
                           (actuals['item_id'] == item_id)]['rating'].values
            if len(actual) > 0:
                absolute_errors.append(abs(predicted - actual[0]))
        
        if absolute_errors:
            return np.mean(absolute_errors)
        return float('inf')
    
    @staticmethod
    def precision_at_k(recommendations: List[int], relevant_items: List[int], k: int) -> float:
        """Calculate Precision@K."""
        
        if k == 0:
            return 0.0
        
        recommendations_at_k = recommendations[:k]
        relevant_and_recommended = set(recommendations_at_k) & set(relevant_items)
        
        return len(relevant_and_recommended) / k
    
    @staticmethod
    def recall_at_k(recommendations: List[int], relevant_items: List[int], k: int) -> float:
        """Calculate Recall@K."""
        
        if len(relevant_items) == 0:
            return 0.0
        
        recommendations_at_k = recommendations[:k]
        relevant_and_recommended = set(recommendations_at_k) & set(relevant_items)
        
        return len(relevant_and_recommended) / len(relevant_items)
    
    @staticmethod
    def ndcg_at_k(recommendations: List[int], relevant_items: List[int], k: int) -> float:
        """Calculate Normalized Discounted Cumulative Gain@K."""
        
        def dcg_at_k(r, k):
            r = np.asfarray(r)[:k]
            if r.size:
                return r[0] + np.sum(r[1:] / np.log2(np.arange(2, r.size + 1)))
            return 0.
        
        recommendations_at_k = recommendations[:k]
        relevance = [1 if item in relevant_items else 0 for item in recommendations_at_k]
        
        # Calculate DCG
        dcg = dcg_at_k(relevance, k)
        
        # Calculate IDCG
        ideal_relevance = [1] * min(len(relevant_items), k)
        ideal_relevance.extend([0] * (k - len(ideal_relevance)))
        idcg = dcg_at_k(ideal_relevance, k)
        
        if idcg == 0:
            return 0.0
        
        return dcg / idcg
    
    @staticmethod
    def coverage(recommendations: List[List[int]], catalog_size: int) -> float:
        """Calculate catalog coverage."""
        
        recommended_items = set()
        for rec_list in recommendations:
            recommended_items.update(rec_list)
        
        return len(recommended_items) / catalog_size
    
    @staticmethod
    def diversity(recommendations: List[int], item_similarity_matrix: np.ndarray) -> float:
        """Calculate diversity of recommendations."""
        
        if len(recommendations) < 2:
            return 0.0
        
        diversity_scores = []
        for i in range(len(recommendations)):
            for j in range(i + 1, len(recommendations)):
                sim = item_similarity_matrix[recommendations[i], recommendations[j]]
                diversity_scores.append(1 - sim)
        
        return np.mean(diversity_scores)
    
    @staticmethod
    def novelty(recommendations: List[int], item_popularity: Dict[int, float]) -> float:
        """Calculate novelty of recommendations."""
        
        novelty_scores = []
        for item in recommendations:
            if item in item_popularity:
                # Novelty is inversely related to popularity
                novelty_scores.append(-np.log2(item_popularity[item] + 1e-10))
        
        return np.mean(novelty_scores) if novelty_scores else 0.0

## 7. Example Usage and Demonstrations

In [ ]:
def generate_sample_data(n_users: int = 1000, n_items: int = 500, 
                        n_interactions: int = 10000) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Generate sample recommendation data."""
    
    np.random.seed(42)
    
    # Generate interactions
    interactions = []
    for _ in range(n_interactions):
        user_id = np.random.randint(1, n_users + 1)
        item_id = np.random.randint(1, n_items + 1)
        rating = np.random.choice([1, 2, 3, 4, 5], p=[0.1, 0.1, 0.2, 0.3, 0.3])
        interactions.append({
            'user_id': user_id,
            'item_id': item_id,
            'rating': rating,
            'timestamp': np.random.randint(1000000, 2000000)
        })
    
    interactions_df = pd.DataFrame(interactions)
    interactions_df = interactions_df.drop_duplicates(['user_id', 'item_id'])
    
    # Generate item features
    categories = ['Action', 'Comedy', 'Drama', 'Sci-Fi', 'Romance']
    item_features = []
    for item_id in range(1, n_items + 1):
        item_features.append({
            'item_id': item_id,
            'category': np.random.choice(categories),
            'popularity': np.random.random(),
            'year': np.random.randint(1990, 2024),
            'description': f"Item {item_id} description with various keywords and features"
        })
    
    item_features_df = pd.DataFrame(item_features)
    
    return interactions_df, item_features_df


def demo_recommender_systems():
    """Demonstrate various recommender systems."""
    
    print("=" * 50)
    print("Recommender Systems Demo")
    print("=" * 50)
    
    # Generate sample data
    print("\nGenerating sample data...")
    interactions, item_features = generate_sample_data(
        n_users=500, n_items=200, n_interactions=5000
    )
    
    # Split data
    train_interactions, test_interactions = train_test_split(
        interactions, test_size=0.2, random_state=42
    )
    
    print(f"Train interactions: {len(train_interactions)}")
    print(f"Test interactions: {len(test_interactions)}")
    print(f"Number of users: {interactions['user_id'].nunique()}")
    print(f"Number of items: {interactions['item_id'].nunique()}")
    
    # 1. Collaborative Filtering
    print("\n" + "="*30)
    print("1. Collaborative Filtering")
    print("="*30)
    
    cf_model = CollaborativeFiltering(method='item_based')
    cf_model.fit(train_interactions)
    
    user_id = 1
    recommendations = cf_model.recommend(user_id, n_recommendations=5)
    print(f"\nRecommendations for user {user_id}:")
    for item_id, score in recommendations:
        print(f"  Item {item_id}: {score:.3f}")
    
    # 2. Content-Based Filtering
    print("\n" + "="*30)
    print("2. Content-Based Filtering")
    print("="*30)
    
    cb_model = ContentBasedFiltering()
    cb_model.fit(item_features, train_interactions)
    
    item_id = 10
    similar_items = cb_model.recommend_similar_items(item_id, n_recommendations=5)
    print(f"\nItems similar to item {item_id}:")
    for sim_item_id, score in similar_items:
        print(f"  Item {sim_item_id}: {score:.3f}")
    
    # 3. Matrix Factorization
    print("\n" + "="*30)
    print("3. Matrix Factorization")
    print("="*30)
    
    mf_model = MatrixFactorization(method='svd', n_factors=20)
    mf_model.fit(train_interactions)
    
    recommendations = mf_model.recommend(user_id, n_recommendations=5)
    print(f"\nRecommendations for user {user_id}:")
    for item_id, score in recommendations:
        print(f"  Item {item_id}: {score:.3f}")
    
    # 4. Hybrid System
    print("\n" + "="*30)
    print("4. Hybrid Recommender System")
    print("="*30)
    
    hybrid_model = HybridRecommenderSystem()
    hybrid_model.fit(
        train_interactions, 
        item_features,
        fit_deep=False  # Skip deep learning for demo speed
    )
    
    recommendations = hybrid_model.recommend(user_id, n_recommendations=5)
    print(f"\nHybrid recommendations for user {user_id}:")
    for item_id, score in recommendations:
        print(f"  Item {item_id}: {score:.3f}")
    
    # 5. Evaluation
    print("\n" + "="*30)
    print("5. Evaluation Metrics")
    print("="*30)
    
    evaluator = RecommenderEvaluator()
    
    # Get recommendations for test users
    test_users = test_interactions['user_id'].unique()[:10]
    
    for test_user in test_users:
        # Get actual items
        actual_items = test_interactions[test_interactions['user_id'] == test_user]['item_id'].tolist()
        
        # Get recommendations
        recommendations = hybrid_model.recommend(test_user, n_recommendations=10)
        recommended_items = [item_id for item_id, _ in recommendations]
        
        # Calculate metrics
        precision = evaluator.precision_at_k(recommended_items, actual_items, k=5)
        recall = evaluator.recall_at_k(recommended_items, actual_items, k=5)
        ndcg = evaluator.ndcg_at_k(recommended_items, actual_items, k=5)
        
        if test_user == test_users[0]:
            print(f"\nMetrics for user {test_user}:")
            print(f"  Precision@5: {precision:.3f}")
            print(f"  Recall@5: {recall:.3f}")
            print(f"  NDCG@5: {ndcg:.3f}")
    
    print("\nDemo completed successfully!")

# Run demo
if __name__ == "__main__":
    demo_recommender_systems()

## Summary

This comprehensive recommendation system implementation includes:

1. **Collaborative Filtering**: User-based and item-based approaches with multiple similarity metrics
2. **Content-Based Filtering**: TF-IDF and feature-based recommendations
3. **Matrix Factorization**: SVD, NMF, ALS, and BPR implementations
4. **Deep Learning**: Neural Collaborative Filtering with embeddings and MLP
5. **Hybrid Systems**: Weighted, switching, and mixed strategies
6. **Evaluation**: Comprehensive metrics including RMSE, MAE, Precision@K, Recall@K, NDCG@K
7. **Advanced Features**: Cold start handling, explanations, diversity, and novelty

The system is production-ready and can handle various recommendation scenarios with different data types and business requirements.